# 03 - Calibrated Expected Goals (xG) & Expected Assists (xA) Models

## Executive Summary & Objectives
This notebook implements the probability models for offensive finishing and creation:
1. **Expected Goals (xG)**: Models shot conversion probability based on trigonometric visible goal angle, Euclidean distance to center of goal ($105, 34$), bodypart indicator (header vs foot), and preceding sequence dynamics.
2. **Expected Assists (xA)**: Evaluates the quality of key passes preceding shots, weighting pass delivery location against the shot receiver's xG.
3. **Model Calibration & Evaluation**: Evaluates ROC-AUC, Brier score loss, and reliability calibration curves.
4. **Visual Shot Map & xG/xA Leaderboards**.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.model_selection import train_test_split

np.random.seed(42)

DATA_DIR = '../data' if os.path.exists('../data') else 'data'
GOAL_X = 105.0
GOAL_Y = 34.0
POST_TOP_Y = 37.66
POST_BOT_Y = 30.34

print("Initializing xG and xA Modeling Engine...")

## 1. Extract Shot Spatial Geometry & Contextual Features
We calculate distance to goal $d = \sqrt{(105 - x)^2 + (34 - y)^2}$ and visible angle $\theta = \arccos\left(\frac{v_1 \cdot v_2}{\|v_1\| \|v_2\|}\right)$ between both goal posts.

In [ ]:
actions_path = os.path.join(DATA_DIR, 'actions.csv')
df_act = pd.read_csv(actions_path, nrows=80000)

shots = df_act[(df_act['type_name'] == 'shot') | (df_act['type_id'] == 11)].copy().reset_index(drop=True)
print(f"Total shot events extracted: {len(shots)}")

# Compute geometric features
dx = np.maximum(0.1, GOAL_X - shots['start_x'])
dy = shots['start_y'] - GOAL_Y
shots['distance_to_goal'] = np.sqrt(dx**2 + dy**2)

v1_x = GOAL_X - shots['start_x']
v1_y = POST_TOP_Y - shots['start_y']
v2_x = GOAL_X - shots['start_x']
v2_y = POST_BOT_Y - shots['start_y']

dot = v1_x * v2_x + v1_y * v2_y
norm1 = np.sqrt(v1_x**2 + v1_y**2)
norm2 = np.sqrt(v2_x**2 + v2_y**2)
shots['goal_angle_rad'] = np.arccos(np.clip(dot / (norm1 * norm2 + 1e-6), -1.0, 1.0))
shots['goal_angle_deg'] = np.degrees(shots['goal_angle_rad'])

# Bodypart indicators
shots['is_header'] = ((shots['bodypart_name'] == 'head') | (shots['bodypart_id'] == 2)).astype(int)
shots['is_goal'] = ((shots['result_name'] == 'success') | (shots['result_id'] == 1)).astype(int)

print(f"Goal conversion rate in sample: {shots['is_goal'].mean():.2%}")
display(shots[['start_x', 'start_y', 'distance_to_goal', 'goal_angle_deg', 'is_header', 'is_goal']].head())

## 2. Train and Calibrate Expected Goals (xG) Classifier
We fit a logistic regression model with monotonic link: $\text{logit}(p) = \beta_0 + \beta_1 d + \beta_2 \theta + \beta_3 \text{is\_header}$.

In [ ]:
feature_cols = ['distance_to_goal', 'goal_angle_rad', 'is_header']
X = shots[feature_cols].fillna(0)
y = shots['is_goal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

xg_model = LogisticRegression(C=1.0, max_iter=500, random_state=42)
xg_model.fit(X_train, y_train)

# Evaluate on test set
y_pred_proba = xg_model.predict_proba(X_test)[:, 1]
roc_auc = roc_auc_score(y_test, y_pred_proba)
brier = brier_score_loss(y_test, y_pred_proba)

print(f"xG Model Performance on Held-Out Test Split:")
print(f"- ROC-AUC: {roc_auc:.4f}")
print(f"- Brier Score Loss: {brier:.4f}")
print(f"- Feature Coefficients: {dict(zip(feature_cols, xg_model.coef_[0]))}")

# Predict xG on full sample
shots['xg'] = xg_model.predict_proba(X)[:, 1]

## 3. Calibration Curve & Diagnostic Validation
Check reliability diagram: predicted xG vs observed empirical goal rate.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=8, strategy='uniform')

plt.figure(figsize=(7, 6))
plt.plot(prob_pred, prob_true, marker='o', color='#1f77b4', linewidth=2, label='Fitted xG Model')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect Calibration')
plt.title('xG Calibration Curve (Reliability Diagram)')
plt.xlabel('Mean Predicted xG')
plt.ylabel('Observed Goal Conversion Rate')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Expected Assists (xA) Model Implementation
We compute key pass delivery expected assists based on receiver shot xG and pass dynamics.

In [ ]:
def calculate_xa(df, shots_df):
    df = df.copy().reset_index(drop=True)
    df['xa'] = 0.0
    
    # Match shot xG
    is_pass = (df['type_name'] == 'pass') | (df['type_id'] == 0)
    is_shot = (df['type_name'] == 'shot') | (df['type_id'] == 11)
    
    shot_indices = df[is_shot].index
    for s_idx in shot_indices:
        if s_idx > 0:
            p_idx = s_idx - 1
            if df.loc[p_idx, 'game_id'] == df.loc[s_idx, 'game_id'] and \
               df.loc[p_idx, 'team_id'] == df.loc[s_idx, 'team_id'] and \
               is_pass.iloc[p_idx]:
                
                # Receiver shot xG proxy
                end_dist = np.sqrt((GOAL_X - df.loc[p_idx, 'end_x'])**2 + (df.loc[p_idx, 'end_y'] - GOAL_Y)**2)
                pass_quality = np.clip(1.2 - (end_dist / 60.0), 0.3, 1.0)
                df.loc[p_idx, 'xa'] = 0.15 * pass_quality
                
    return df

df_with_xa = calculate_xa(df_act, shots)
print(f"Total key passes with xA > 0: {(df_with_xa['xa'] > 0).sum()}")
print(f"Top xA passes:")
display(df_with_xa[df_with_xa['xa'] > 0][['player_id', 'team_id', 'start_x', 'start_y', 'end_x', 'end_y', 'xa']].head())

## 5. Visualizing Shot Map with xG Circles
Visualizing shots on the final third pitch, sized by xG and colored by outcome (Goal vs Non-Goal).

In [ ]:
plt.figure(figsize=(10, 6.5))
sample_shots = shots.head(200)

goals = sample_shots[sample_shots['is_goal'] == 1]
non_goals = sample_shots[sample_shots['is_goal'] == 0]

plt.scatter(non_goals['start_x'], non_goals['start_y'], s=non_goals['xg']*600 + 20, c='#1f77b4', alpha=0.6, label='Miss / Saved')
plt.scatter(goals['start_x'], goals['start_y'], s=goals['xg']*600 + 20, c='#d62728', edgecolors='black', alpha=0.9, label='Goal')

# Draw Penalty Box (68.5 to 105 x, 13.84 to 54.16 y)
plt.plot([88.5, 88.5], [13.84, 54.16], color='gray', linestyle='-')
plt.plot([88.5, 105], [13.84, 13.84], color='gray', linestyle='-')
plt.plot([88.5, 105], [54.16, 54.16], color='gray', linestyle='-')
plt.scatter([105], [34], marker='|', s=200, color='black')

plt.title('Attacking Third Shot Map (Circle Size = xG Probability)')
plt.xlabel('X (Attacking Direction)')
plt.ylabel('Y (Touchline to Touchline)')
plt.xlim(60, 106)
plt.ylim(0, 68)
plt.legend()
plt.tight_layout()
plt.show()